# General Forward

Generate feasible data in the notebook, then train the general surrogate model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from kkthn import KKTHardNet

TRAIN = {
    "epochs": 2,
    "batch_size": 8,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 16,
    "hidden_layers": 1,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}


## Generate Example Data

In [ ]:
data_dir = Path('notebooks/data/general_forward')
data_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(4)
y1 = rng.uniform(0.0, 1.0, size=32)
y3 = rng.uniform(-1.0, 1.0, size=32)
mask = y1**2 + y3**2 <= 1.6
y1 = y1[mask][:32]
y3 = y3[mask][:32]
y2 = rng.uniform(-0.5, 0.5, size=y1.size)
a0, a1 = 1.0, -1.0
x1 = a0 * y1 + y2
x2 = y2 - a1 * y3

pd.DataFrame({'x1': x1, 'x2': x2}).to_csv(data_dir / 'parameters.csv', index=False)
pd.DataFrame({'y1': y1, 'y2': y2, 'y3': y3}).to_csv(data_dir / 'variables.csv', index=False)
data_dir


## Build And Train

In [ ]:
model = KKTHardNet(name='general_forward', train=TRAIN)
x = model.add_parameter(['x1', 'x2'])
y = model.add_variable(['y1', 'y2', 'y3'])
model.constraints.add(
    y.y1 + y.y2 - x.x1 == 0,
    y.y2 + y.y3 - x.x2 == 0,
    y.y1**2 + y.y3**2 <= 2.0,
    y.y1 >= 0,
)
model.dataset(parameters=data_dir / 'parameters.csv', variables=data_dir / 'variables.csv')
result = model.model()
result['metadata_path']


In [ ]:
model = KKTHardNet(name='general_forward')
model.load('/mnt/research/Hasan_Faruque/Shared/kkt-hardnet/notebooks/general_forward_20260414_172033/metadata.json')

In [ ]:
model.predict([1.0, 100])